In [1]:
import subprocess, os
CODE   = "/kaggle/input/datasets/theokorir/knee-mri-pipeline/knee_pipeline"
LABELS = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/labels_soft.csv"
C160   = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-caching/cache"
C224   = "//kaggle/input/notebooks/theokorir/knee-mri-cache224/cache224" 

def running(out):
    return subprocess.run(["pgrep", "-f", "--", f"--out {out}"], capture_output=True).returncode == 0

def launch(gpu, name, cache, extra):
    out = f"/kaggle/working/{name}"
    if not os.path.isdir(cache): print(name, "SKIP: cache missing", flush=True); return None
    if running(out):             print(name, "SKIP: already running", flush=True); return None
    cmd = ["python", f"{CODE}/train.py", "--cache_dir", cache, "--labels", LABELS, "--epochs", "12",
           "--pretrained", "--workers", "2", "--train_slices", "16", "--only_folds", "0", "--out", out] + extra
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    return subprocess.Popen(cmd, cwd=CODE, env=env, stdout=open(f"{out}.log", "w"), stderr=subprocess.STDOUT)

base = ["--bs", "4", "--accum", "2"]

# GPU 0: 224 px candidate (uses train.py's default seed, like the baseline)
p224 = launch(0, "exp_224", C224, ["--bs", "2", "--accum", "4"])

# GPU 1: seed repeats at 160 px, back to back
p7 = launch(1, "exp_seed7", C160, base + ["--seed", "7"])
if p7: p7.wait(); print("exp_seed7 exit code", p7.returncode, flush=True)
p11 = launch(1, "exp_seed11", C160, base + ["--seed", "11"])
if p11: p11.wait(); print("exp_seed11 exit code", p11.returncode, flush=True)

# the notebook must wait on every subprocess or the commit kills them
if p224: p224.wait(); print("exp_224 exit code", p224.returncode, flush=True)

exp_seed7 exit code 0
exp_seed11 exit code 0
exp_224 exit code 0


In [2]:
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

BASE = "/kaggle/input/notebooks/theokorir/knee-mri-diagnosis-training/oof_all.csv"
EXP = {
    "seed7":  "/kaggle/working/exp_seed7/oof.csv",
    "seed11": "/kaggle/working/exp_seed11/oof.csv",
    "e224":   "/kaggle/working/exp_224/oof.csv",
}
NOISE_RUNS = ["baseline", "seed7", "seed11"]      # e224 is deliberately excluded
KEY = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"]
ID = "StudyInstanceUID"

b_all = pd.read_csv(BASE).set_index(ID)
LAB = [c[2:] for c in b_all.columns if c.startswith("p_")]

raw = {}
for n, p in EXP.items():
    if not os.path.exists(p):
        print(f"{n}: no oof.csv (run missing or not finished)"); continue
    raw[n] = pd.read_csv(p).set_index(ID)

# fold-0 study set
if "fold" in b_all.columns:
    f0 = b_all.index[b_all.fold == 0]
elif raw:
    first = next(iter(raw.values()))
    f0 = first.index[first[f"p_{LAB[0]}"].notna()]
else:
    raise SystemExit("No fold column in baseline and no experiment oof.csv to infer fold 0 from.")

b = b_all.loc[f0]
runs = {"baseline": b}
for n, e in raw.items():
    e = e.loc[b.index]
    if e[f"p_{LAB[0]}"].isna().any():
        print(f"{n}: WARNING, NaN predictions in fold 0, skipped"); continue
    runs[n] = e

def auc(d, l, hi=0.9, lo=0.1):
    y = d[f"y_{l}"]; m = (y >= hi) | (y <= lo)
    yy = (y[m] >= hi).astype(int)
    return roc_auc_score(yy, d.loc[m, f"p_{l}"]) if yy.nunique() == 2 else np.nan

T = pd.DataFrame({n: {l: auc(d, l) for l in LAB} for n, d in runs.items()})
T.loc["MACRO (12)"] = T.loc[LAB].mean()
T.loc["KEY-4 mean"] = T.loc[KEY].mean()
print(T.round(3))

# baseline sanity check
mac, k4 = T.loc["MACRO (12)", "baseline"], T.loc["KEY-4 mean", "baseline"]
print(f"\nBaseline check: macro {mac:.4f} (expect ~0.7795), KEY-4 {k4:.4f} (expect ~0.7547)")
if abs(mac - 0.7795) > 0.003 or abs(k4 - 0.7547) > 0.003:
    print("!! Baseline does not reproduce. Check thresholds / study subset before trusting anything below.")

# noise floor from seed repeats
nr = [n for n in NOISE_RUNS if n in T.columns]
floor = None
if len(nr) >= 2:
    k = T.loc["KEY-4 mean", nr]; m = T.loc["MACRO (12)", nr]
    floor = k.max() - k.min()
    plr = T.loc[LAB, nr].max(axis=1) - T.loc[LAB, nr].min(axis=1)
    print(f"\nNoise floor over {nr}")
    print(f"  KEY-4 range: {floor:.4f}   macro range: {m.max()-m.min():.4f}")
    print(f"  per-label range: median {plr.median():.3f}, max {plr.max():.3f}")
else:
    print("\nNoise floor: need at least one finished seed run besides the baseline.")

# 224 px verdict
if "e224" in T.columns:
    gain = T.loc["KEY-4 mean", "e224"] - T.loc["KEY-4 mean", "baseline"]
    wins = int((T.loc[KEY, "e224"] > T.loc[KEY, "baseline"]).sum())
    print(f"\n224 px vs baseline: KEY-4 gain {gain:+.4f}, macro gain "
          f"{T.loc['MACRO (12)','e224'] - T.loc['MACRO (12)','baseline']:+.4f}, better on {wins}/4 key labels")
    if floor is not None:
        print(f"  noise floor {floor:.4f} -> " + ("clearly above noise" if gain > 2*floor else "NOT clearly above noise"))
    print("  spec target: gain >= ~0.03, or gains on all four key labels")

# paired bootstrap on KEY-4 (study sampling only, not seed noise)
def key4(d, idx): return np.nanmean([auc(d.iloc[idx], l) for l in KEY])
rng = np.random.default_rng(0); N = len(b)
boots = [rng.integers(0, N, N) for _ in range(200)]
for n, d in runs.items():
    if n == "baseline": continue
    diff = [key4(d, i) - key4(b, i) for i in boots]
    print(f"{n:9s} KEY-4 diff vs baseline: {np.mean(diff):+.3f}  90% CI [{np.percentile(diff,5):+.3f}, {np.percentile(diff,95):+.3f}]")

                  baseline  seed7  seed11   e224
ACL                  0.833  0.840   0.847  0.803
MCL                  0.713  0.730   0.717  0.715
Medial Meniscus      0.757  0.756   0.763  0.765
Lateral Meniscus     0.715  0.713   0.710  0.712
Medial OA            0.861  0.861   0.866  0.858
Lateral OA           0.749  0.736   0.746  0.743
PF OA                0.722  0.731   0.728  0.717
Effusion             0.801  0.797   0.812  0.809
Synovitis            0.799  0.800   0.795  0.783
Baker's              0.747  0.730   0.751  0.760
Contusion            0.808  0.801   0.807  0.771
Fracture             0.848  0.857   0.853  0.795
MACRO (12)           0.779  0.779   0.783  0.769
KEY-4 mean           0.755  0.760   0.759  0.749

Baseline check: macro 0.7795 (expect ~0.7795), KEY-4 0.7547 (expect ~0.7547)

Noise floor over ['baseline', 'seed7', 'seed11']
  KEY-4 range: 0.0050   macro range: 0.0035
  per-label range: median 0.009, max 0.021

224 px vs baseline: KEY-4 gain -0.0059, macro gai